# Tratamento do listings.csv   

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_raw = pd.read_csv('listings.csv')

print(f'Total de registros originais: {len(df_raw)}')
print(f'Tipo de coluna \'price\': {df_raw["price"].dtype}')
print('\nPrimeiros 5 valores de price na base bruta: ')
print(df_raw['price'].head())

Total de registros originais: 42378
Tipo de coluna 'price': str

Primeiros 5 valores de price na base bruta: 
0    540,00
1    289,00
2    309,00
3    225,00
4    244,00
Name: price, dtype: str


In [8]:
# Cria uma cópia do DataFrame bruto
df_listings = df_raw.copy()

# Limpeza ajustada para formato regional (ex: R$ 1.250,00 ou 289,00)
df_listings['price'] = (
    df_listings['price']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.replace('.', '', regex=False)   # Remove o ponto de milhar (ex: 1.250,00 -> 1250,00)
    .str.replace(',', '.', regex=False)  # Troca a vírgula dos centavos por ponto para o Python (1250,00 -> 1250.00)
    .str.strip()
)

# Conversão explícita para número decimal (float)
df_listings['price'] = pd.to_numeric(df_listings['price'], errors='coerce')

# Remove registros onde o preço não pôde ser convertido ou é nulo
df_listings = df_listings.dropna(subset=['price']).copy()

# Remoção de valores irreais e corte no percentil 99
df_listings = df_listings[df_listings['price'] >= 20].copy()
p99 = df_listings['price'].quantile(0.99)
df_listings_clean = df_listings[df_listings['price'] <= p99].copy()

df_listings_clean.to_csv('listings_cleaned.csv', index=False)

print(f"Total de registros após limpeza: {len(df_listings_clean)}")
print(f"Corte P99 aplicado: R$ {p99:.2f}")
print("\nVerifique se os preços agora estão corretos:")
print(df_listings_clean['price'].head())

Total de registros após limpeza: 41432
Corte P99 aplicado: R$ 1911.50

Verifique se os preços agora estão corretos:
0    540.0
1    289.0
2    309.0
3    225.0
4    244.0
Name: price, dtype: float64


# Tratamento de nulos

In [7]:
#Imputação de valores nulos na coluna reviews_per_month, caso exista
if 'reviews_per_month' in df_listings_clean.columns:
    df_listings_clean['reviews_per_month'] = df_listings_clean['reviews_per_month'].fillna(0)

#Higienização de strings dos bairros
df_listings_clean['neighbourhood'] = df_listings_clean['neighbourhood'].astype(str).str.strip()

#Exportar base limpa
df_listings_clean.to_csv('listings_clean.csv', index=False)

#Resumo estatístico para validação
df_listings_clean['price'].describe()

count    41432.000000
mean       370.777032
std        219.174862
min         26.000000
25%        249.000000
50%        330.000000
75%        425.000000
max       1911.000000
Name: price, dtype: float64

# Tratamento de Sazonalidade

In [10]:
#Carregando base de reviews
df_reviews = pd.read_csv('reviews.csv', usecols=['listing_id', 'date'])
df_reviews['date'] = pd.to_datetime(df_reviews['date'])

# Engenharia de atributos temporais
df_reviews['review_year'] = df_reviews['date'].dt.year
df_reviews['review_month'] = df_reviews['date'].dt.month
df_reviews['year_month'] = df_reviews['date'].dt.to_period('M').astype(str)

#Filtar histórico recente e salvar
df_reviews_clean = df_reviews[df_reviews['review_year'] >= 2018].copy()
df_reviews_clean.to_csv('reviews_clean.csv', index=False)

print(f'Total de registros de reviews após limpeza: {len(df_reviews_clean)}')

Total de registros de reviews após limpeza: 1554409
